**Lab type:** review  
**Course:** NL301 Natural Language Processing with Python  
**Lesson:** 10 — Evaluating NLP Models  
**Task:** Review NLP evaluation code and answer five questions about metric selection and common pitfalls.

## Setup

In [ ]:
!pip install nltk evaluate bert-score sacrebleu --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
import numpy as np


## The evaluation code to review

In [ ]:
# Summarisation output to evaluate
references = [
    "The model exceeded analyst expectations beating forecasts by a wide margin.",
    "Scientists discovered a new species of deep sea fish near the Pacific trench.",
]
hypotheses = [
    "The company beat forecasts significantly.",
    "Researchers found an unknown fish species in the Pacific Ocean.",
]

# Evaluation code under review — read before answering the questions below
scores = []
for ref, hyp in zip(references, hypotheses):
    score = sentence_bleu(ref, hyp)     # (A) reference is a string, not list of lists
    scores.append(score)

print("Per-sentence BLEU scores:", scores)
print("Mean BLEU:", np.mean(scores))


---
## Review Question 1: sentence_bleu requires list of lists

`sentence_bleu(reference_string, hypothesis)` passes a string as the reference. NLTK iterates over characters, not words — BLEU = 0.0 silently. Demonstrate and fix.

In [ ]:
# Demonstrate the silent failure
ref_string = "The model exceeded analyst expectations beating forecasts"
hyp        = "The company beat forecasts significantly"

score_wrong = sentence_bleu(ref_string, hyp.split())
print(f"Wrong (string reference):  {score_wrong:.4f}")

# Fix: reference must be a list of tokenised references — list of lists
ref_correct = [ref_string.split()]        # [['The', 'model', ...]]
score_correct = sentence_bleu(ref_correct, hyp.split(),
                               smoothing_function=SmoothingFunction().method1)
print(f"Correct (list of lists):   {score_correct:.4f}")
# Explain why sentence_bleu expects list of lists (multiple reference translations allowed)


---
## Review Question 2: Macro F1 vs weighted F1 for rare entities

NER dataset: 3 PER, 1 ORG, 100 O tokens. Which metric surfaces performance on rare entity types?

In [ ]:
from sklearn.metrics import f1_score, classification_report

# Simulated NER predictions (O = outside, PER = person, ORG = organisation)
true_labels = ['O']*95 + ['PER','PER','PER','ORG'] + ['O']*5
pred_labels = ['O']*95 + ['PER','PER','O',  'O']   + ['O']*5   # misses last PER and ORG

labels = ['O', 'PER', 'ORG']
print(classification_report(true_labels, pred_labels, labels=labels))
print("Macro F1:   ", f1_score(true_labels, pred_labels, labels=labels, average='macro'))
print("Weighted F1:", f1_score(true_labels, pred_labels, labels=labels, average='weighted'))
# Which metric better surfaces that ORG was entirely missed?


---
## Review Question 3: BLEU for paraphrase evaluation

BLEU uses n-gram overlap. For paraphrases where meaning is preserved but wording differs, BLEU = 0. When is BLEU inappropriate?

In [ ]:
from bert_score import score as bert_score

reference  = "The company beat forecasts significantly."
hypothesis = "The firm exceeded expectations by a wide margin."

# BLEU
bleu = sentence_bleu([reference.split()], hypothesis.split(),
                     smoothing_function=SmoothingFunction().method1)
print(f"BLEU:      {bleu:.4f}")

# BERTScore
P, R, F1 = bert_score([hypothesis], [reference], lang="en", verbose=False)
print(f"BERTScore F1: {F1[0].item():.4f}")
# Explain: when is lexical overlap (BLEU) insufficient for evaluation?


---
## Review Question 4: ROUGE variants for summarisation

ROUGE-1 vs ROUGE-2 vs ROUGE-L — which is standard for summarisation benchmarks and why?

In [ ]:
# Install rouge-score if not available
try:
    from rouge_score import rouge_scorer
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'rouge-score', '--quiet'])
    from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

ref  = "Scientists discovered a new fish species in the deep Pacific Ocean."
hyp1 = "Researchers found an unknown species of fish near the Pacific trench."   # paraphrase
hyp2 = "New fish species was discovered by scientists in the Pacific."            # reordered

for hyp in [hyp1, hyp2]:
    scores = scorer.score(ref, hyp)
    print(f"Hypothesis: '{hyp[:60]}'")
    for k, v in scores.items():
        print(f"  {k}: P={v.precision:.3f} R={v.recall:.3f} F={v.fmeasure:.3f}")
    print()
# Which ROUGE variant do most summarisation papers report, and why?


---
## Review Question 5: Metric ranking for summarisation

Given a summarisation task, rank BLEU, ROUGE-2, and BERTScore for appropriateness. Justify each ranking.

In [ ]:
# Run all three metrics on the same summarisation example
ref_sum  = "The central bank raised interest rates to combat persistent inflation pressures."
hyp_sum  = "The bank increased rates to fight inflation."

# BLEU
bleu = sentence_bleu([ref_sum.split()], hyp_sum.split(),
                     smoothing_function=SmoothingFunction().method1)

# ROUGE-2
rouge2 = scorer.score(ref_sum, hyp_sum)['rouge2'].fmeasure

# BERTScore
P, R, F1 = bert_score([hyp_sum], [ref_sum], lang="en", verbose=False)

print(f"BLEU:         {bleu:.4f}")
print(f"ROUGE-2:      {rouge2:.4f}")
print(f"BERTScore F1: {F1[0].item():.4f}")
print()
print("Ranking for summarisation (most to least appropriate):")
print("1. ___ — because ___")
print("2. ___ — because ___")
print("3. ___ — because ___")
